# Assignment 9

### Imports

In [2]:
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json

### Function for making train/validation/test - split

In [9]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

### Modified enis data load function to not rely on locally defined data_dir within function

In [10]:
def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined

### Apply to get the dataframes

In [11]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

### Function for splitting input (x, y) and target (z)

In [19]:
def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

### Apply to get X_train, Y_train e.t.c

In [ ]:
x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)

### Functions for model-handling

In [ ]:
# Enis function but it doesn't rely on a global candidates_dir variable
def save_candidate_model(model, model_name, candidates_dir):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


# Enis load_champion function but doesn't rely on global metadata_dir variable
def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None


# Enis save_champion model but function doesn't rely on globally defined dirs
def save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# Enis update_champion but with local variable dirs
def update_champion(metadata_dir, champion_dir, model, model_name, mse, mae, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters)

    elif mae < current["mae"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")

### ML

In [ ]:
# Define paths
model_dir = "../assignment9_models"
candidates_dir = os.path.join(model_dir, "candidates")
champion_dir = os.path.join(model_dir, "champion")
metadata_dir = os.path.join(model_dir, "metadata")

# Define search space 
search_space = [{"model_type": "dense",
                 "hidden_layers": [1, 2, 3],
                 "activation": "relu",
                 "optimizer": "adam",
                 "lr": 0.001,
                 "batch_size": 32,
                 "epochs": 50
                 }]


# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


# Build model from configuration
def build_model(config, input_size):
    layers = []
    current_dim = input_size

    # Build the layers
    for h in config["hidden_layers"]:
        layers.append(nn.Linear(current_dim, h))

        if config["activation"] == "relu":
            layers.append(nn.ReLU())
        elif config["activation"] == "tanh":
            layers.append(nn.Tanh())
        
        current_dim = h
    
    # Output layer
    layers.append(nn.Linear(current_dim, 1))
    return nn.Sequential(*layers)


# Train model using config
def train_model(model, config, X_train, Y_train):
    criterion = nn.MSELoss()

    if config["optimizer"] == "adam":
        optimizer = optim.Adam(model.parameters(), lr = config["lr"])
    
    elif config["optimizer"] == "sgd":
        optimizer = optim.SGD(model.parameters(), lr = config["lr"])
    else:
        print("Invalid optimizer")
    
    # Training loop
    model.train()
    for epoch in range(config["epochs"]):
        predictions = model(X_train)
        loss = criterion(predictions, Y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Final training metrics
    model.eval()
    with torch.no_grad():
        final_train_loss = criterion(model(X_train), Y_train).item()
    
    # Return trained model and it's training loss
    return model, final_train_loss

